
    # Video Game Popularity Prediction
    
    This notebook contains code to predict whether a video game is popular or not based on certain features such as the publisher, genre, and platform. The prediction is made using various machine learning models, including Support Vector Machine, K-Nearest Neighbors, Decision Tree, Random Forest and XGBoost.
    

In [1]:
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split, RandomizedSearchCV
    from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
    from sklearn.svm import SVC
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    from sklearn.feature_selection import SelectFromModel
    from imblearn.combine import SMOTEENN
    import xgboost as xgb
    from sklearn.linear_model import LogisticRegression
    import time


In [2]:

    # Timer to measure runtime
    start_time = time.time()

    print("Step 1: Loading the dataset... [0% completed]")
    df = pd.read_csv('/content/all_video_games(cleaned).csv')


Step 1: Loading the dataset... [0% completed]


FileNotFoundError: [Errno 2] No such file or directory: '/content/all_video_games(cleaned).csv'

In [ ]:
import os

file_path = '/content/all_video_games(cleaned).csv'
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' does NOT exist. Please ensure it is uploaded or the path is correct.")

In [ ]:

    print("Step 2: Displaying dataset columns and initial values... [5% completed]")
    print("Columns in the dataset:")
    print(df.columns)
    print("\nFirst 5 rows of the dataset:")
    print(df.head())


In [ ]:

    print("Step 3: Performing Feature Engineering... [10% completed]")
    df['Release Year'] = pd.to_datetime(df['Release Date'], errors='coerce').dt.year.fillna(0).astype(int)
    df['User Score'] = df['User Score'].fillna(df['User Score'].median())
    df['User Ratings Count'] = df['User Ratings Count'].fillna(df['User Ratings Count'].median())
    df['Publisher'] = df['Publisher'].fillna('Unknown')

    # Group genres into broader categories
    genre_mapping = {
        'Action': 'Action', 'Action RPG': 'RPG', 'Western RPG': 'RPG', 'JRPG': 'RPG',
        'Turn-Based Strategy': 'Strategy', 'Real-Time Strategy': 'Strategy',
        'FPS': 'Shooter', 'Third Person Shooter': 'Shooter', 'Puzzle': 'Puzzle',
        'Sports': 'Sports', 'Racing': 'Racing', 'Arcade Racing': 'Racing',
        'Adventure': 'Adventure', 'Simulation': 'Simulation', 'Tycoon': 'Simulation',
    }
    df['Genres Grouped'] = df['Genres'].map(genre_mapping).fillna('Other')

    # Extracting platform information from 'Platforms Info' column
    def extract_platforms(platform_info):
        try:
            platforms = eval(platform_info)
            if isinstance(platforms, list) and len(platforms) > 0:
                return platforms[0]['Platform']
        except:
            return 'Unknown'
        return 'Unknown'

    df['Platform'] = df['Platforms Info'].apply(extract_platforms)


In [ ]:

    # Check if 'Platform' column exists before encoding
    if 'Platform' in df.columns:
        print("Step 4: Encoding categorical variables... [20% completed]")
        le_publisher = LabelEncoder()
        df['Publisher'] = le_publisher.fit_transform(df['Publisher'])

        le_genres = LabelEncoder()
        df['Genres Grouped'] = le_genres.fit_transform(df['Genres Grouped'])

        le_platform = LabelEncoder()
        df['Platform'] = le_platform.fit_transform(df['Platform'])

        df = df[['Publisher', 'User Score', 'User Ratings Count', 'Release Year', 'Genres Grouped', 'Platform']]

        print("Step 5: Splitting features and target... [30% completed]")
        X = df[['Publisher', 'User Score', 'User Ratings Count', 'Release Year', 'Platform']]
        y = df['Genres Grouped']
    else:
        print("Error: 'Platform' column not found in the dataset.")
        exit()


In [ ]:

    print("Step 6: Applying SMOTEENN for class balancing... [40% completed]")
    smoteenn = SMOTEENN(random_state=42)
    X_resampled, y_resampled = smoteenn.fit_resample(X, y)


In [ ]:

    print("Step 7: Standardizing features... [50% completed]")
    scaler = StandardScaler()
    X_resampled = scaler.fit_transform(X_resampled)


In [ ]:

    print("Step 8: Adding polynomial features... [60% completed]")
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_resampled = poly.fit_transform(X_resampled)


In [ ]:

    print("Step 9: Splitting data into train and test sets... [70% completed]")
    X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)


In [ ]:

    # Feature Selection
    print("Step 10: Performing Feature Selection... [75% completed]")
    feature_selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42))
    X_train = feature_selector.fit_transform(X_train, y_train)
    X_test = feature_selector.transform(X_test)


In [ ]:

    # Evaluation Function
    def evaluate_model(model_name, model):
        print(f"Training {model_name}... [in progress]")
        model.fit(X_train, y_train)
        print(f"Evaluating {model_name}... [in progress]")
        y_pred = model.predict(X_test)
        print(f"\nEvaluation Report for {model_name}:\n")
        print("Accuracy Score:", accuracy_score(y_test, y_pred))
        print("Classification Report:\n", classification_report(y_test, y_pred))
        print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
        print(f"Finished {model_name}!\n{'-' * 50}")


In [ ]:

    print("Step 11: Training and evaluating Support Vector Machine... [80% completed]")
    svc = SVC(C=1000, kernel='rbf', gamma=0.0001, class_weight='balanced', random_state=42)
    evaluate_model("Support Vector Machine", svc)


In [ ]:

    print("Step 12: Training and evaluating K-Nearest Neighbors... [85% completed]")
    knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
    evaluate_model("K-Nearest Neighbors", knn)


In [ ]:

    print("Step 13: Training and evaluating Decision Tree... [90% completed]")
    dt = DecisionTreeClassifier(max_depth=50, class_weight='balanced', random_state=42)
    evaluate_model("Decision Tree", dt)


In [ ]:

    print("Step 14: Training and evaluating Random Forest... [95% completed]")
    rf = RandomForestClassifier(n_estimators=1000, max_depth=50, class_weight='balanced', random_state=42, n_jobs=-1)
    evaluate_model("Random Forest", rf)


In [ ]:

    print("Step 15: Training and evaluating XGBoost... [100% completed]")
    xgb_model = xgb.XGBClassifier(max_depth=15, learning_rate=0.01, n_estimators=1000, random_state=42, eval_metric='mlogloss', n_jobs=-1)
    evaluate_model("XGBoost", xgb_model)


In [ ]:

    def predict_game_popularity(genre, publisher, platform, user_score, user_ratings_count):
      # Encoding the input features
      genre_encoded = le_genres.transform([genre])[0]
      publisher_encoded = le_publisher.transform([publisher])[0]
      platform_encoded = le_platform.transform([platform])[0]

      # Creating the feature array with actual values for user_score and user_ratings_count
      input_features = np.array([[publisher_encoded, user_score, user_ratings_count, 0, platform_encoded]])
      input_df = pd.DataFrame(input_features, columns=['Publisher', 'User Score', 'User Ratings Count', 'Release Year', 'Platform'])

      # Aligning the input DataFrame with the training DataFrame to avoid feature name warnings
      input_df = input_df[['Publisher', 'User Score', 'User Ratings Count', 'Release Year', 'Platform']]

      # Standardizing and transforming the input features
      input_features = scaler.transform(input_df)
      input_features = poly.transform(input_features)
      input_features = feature_selector.transform(input_features)

      # Making predictions using all the models
      models = {
          "Support Vector Machine": svc,
          "K-Nearest Neighbors": knn,
          "Decision Tree": dt,
          "Random Forest": rf,
          "XGBoost": xgb_model,
      }

      for model_name, model in models.items():
          predicted_popularity = model.predict(input_features)[0]
          popularity_label = 'Popular' if predicted_popularity == 1 else 'Not Popular'
          print(f"{model_name} Prediction: {popularity_label}")

In [ ]:

    # User Input
    genre=input("Enter Genre: ")
    publisher=input("Enter Publisher: ")
    platform=input("Enter Platform: ")
    user_score=float(input("Enter User Score: "))
    user_ratings_count=int(input("Enter User Ratings Count: "))
    predict_game_popularity(genre, publisher, platform, user_score, user_ratings_count)


In [ ]:

    end_time = time.time()
    total_time = (end_time - start_time) / 60
    print(f"\nAll models completed in approximately {total_time:.2f} minutes.")
    print("Process Finished! 🎉")
